In [1]:
import warnings
import tensorflow as tf
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')

import numpy as np
import pandas as pd
import deepchem as dc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_max_pool
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from sklearn import metrics

Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading some Jax models, missing a dependency. No module named 'jax'


In [2]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = GCNConv(30, 256)
        self.conv2 = GCNConv(256, 256)
        self.conv3 = GCNConv(256, 256)
        self.conv4 = GCNConv(256, 256)
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)
        self.dropout1 = nn.Dropout(p=0.4)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        x = self.conv4(x, edge_index)
        x = F.relu(x)
        x = global_max_pool(x, data.batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

In [3]:
def custom_collate(batch):
    data_list, target_list = zip(*batch)
    batch_data = Batch.from_data_list(data_list)
    batch_target = torch.stack(target_list)
    return batch_data, batch_target

In [4]:
def calculate_statistics(group):
    r2_test = group['r2_test']
    r2_test_dict = {f'run{i}': r2_test_val for i, r2_test_val in enumerate(r2_test)}
    return pd.Series({
        **r2_test_dict, 
        'r2_test_mean': np.mean(r2_test),
        'r2_test_max': np.max(r2_test),
        'r2_test_min': np.min(r2_test),
        'r2_test_std': np.std(r2_test, ddof=0),
    })

def calculate_statistics2(group):
    rmse_test = group['rmse_test']
    rmse_test_dict = {f'run{i}': rmse_test_val for i, rmse_test_val in enumerate(rmse_test)}
    return pd.Series({
        **rmse_test_dict, 
        'rmse_test_mean': np.mean(rmse_test),
        'rmse_test_max': np.max(rmse_test),
        'rmse_test_min': np.min(rmse_test),
        'rmse_test_std': np.std(rmse_test, ddof=0),
    })

In [5]:
torch.manual_seed(0)

epochs = 120
lr = 9e-3
wd = 2e-5

results_r2 = []
results_rmse = []
for random_state in range(10):
    torch.manual_seed(0)
    
    for dataset in ["abcgg", "aatsc3d", "atsc3d", "kappa2", "peoevsa6", "bertzct", "ggi10", "vsaestate3",
                    "atsc4i", "bcutp1l", "kappa3", "estatevsa3", "kier3", "aats8p", "kier2", "frnh0"]:
        torch.manual_seed(0)
        
        for t in ["Yield_CN"]:
            torch.manual_seed(0)
            scaler = StandardScaler()
            df = pd.read_csv('data_Real/data_real.csv')
            smiles = df["SMILES"]
            featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
            X = featurizer.featurize(smiles)
            
            y = df[t]
            data_train, data_test, target_train, target_test = train_test_split(X, y, test_size=0.5, random_state=random_state)

            target_train = scaler.fit_transform(target_train.values.reshape(-1, 1)).flatten()
            target_test = scaler.transform(target_test.values.reshape(-1, 1)).flatten()
            
            target_train = torch.tensor(target_train, dtype=torch.float32)
            target_test = torch.tensor(target_test, dtype=torch.float32)

            data_train_list = []
            for graph_data in data_train:
                node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
                edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
                edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
                data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
                data_train_list.append(data)

            data_test_list = []
            for graph_data in data_test:
                node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
                edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
                edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
                data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
                data_test_list.append(data)

            train_loader = DataLoader(list(zip(data_train_list, target_train)), batch_size=len(data_train_list), collate_fn=custom_collate)
            test_loader = DataLoader(list(zip(data_test_list, target_test)), batch_size=len(data_test_list), collate_fn=custom_collate)

            model = Net()
            model.load_state_dict(torch.load(f'data_AI+Random/model_{dataset}_sc.pth'))
            model.fc3 = nn.Linear(128, 1)
        
            model.train()
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
            criterion = nn.MSELoss()
        
            for param in model.conv1.parameters():
                param.requires_grad = False
            for param in model.conv2.parameters():
                param.requires_grad = False
            for param in model.conv3.parameters():
                param.requires_grad = False
            for param in model.conv4.parameters():
                param.requires_grad = False

            device = torch.device('cpu')
            model.to(device)

            for epoch in range(epochs):
                for data, target in train_loader:
                    data = data.to(device)
                    target = target.to(device)
                    optimizer.zero_grad()
                    out = model(data)
                    loss = criterion(out, target.view(-1, 1))
                    loss.backward()
                    optimizer.step()

            model.eval()
            pred_train = []
            for data, target in train_loader:
                data = data.to(device)
                with torch.no_grad():
                    out = model(data)
                pred_train.append(out.cpu().numpy())
            pred_train = np.concatenate(pred_train)

            pred_test = []
            for data, target in test_loader:
                data = data.to(device)
                with torch.no_grad():
                    out = model(data)
                pred_test.append(out.cpu().numpy())
            pred_test = np.concatenate(pred_test)

            pred_train = scaler.inverse_transform(pred_train)
            pred_test = scaler.inverse_transform(pred_test)
            target_train = scaler.inverse_transform(target_train.numpy().reshape(-1, 1)).flatten()
            target_test = scaler.inverse_transform(target_test.numpy().reshape(-1, 1)).flatten()

            r2_test_score = metrics.r2_score(target_test, pred_test)
            rmse_test_score = metrics.root_mean_squared_error(target_test, pred_test)
            results_r2.append({'source': dataset, 'target': t, 'r2_test': r2_test_score})
            results_rmse.append({'source': dataset, 'target': t, 'rmse_test': rmse_test_score})

results_df = pd.DataFrame(results_r2)
gen_results = results_df.groupby(['source', 'target']).apply(calculate_statistics).reset_index()
results_df2 = pd.DataFrame(results_rmse)
gen_results2 = results_df2.groupby(['source', 'target']).apply(calculate_statistics2).reset_index()

In [6]:
gen_results.T.to_csv('result/result_yield_CN_r2.csv', header=False)
gen_results.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
source,aats8p,aatsc3d,abcgg,atsc3d,atsc4i,bcutp1l,bertzct,estatevsa3,frnh0,ggi10,kappa2,kappa3,kier2,kier3,peoevsa6,vsaestate3
target,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN
run0,0.328385,0.161434,0.187002,0.194527,0.45675,0.147392,0.205009,0.374136,0.205607,0.423093,0.229676,0.455024,0.272941,0.522144,0.124578,0.237924
run1,0.161523,0.205478,0.425759,0.259014,0.349935,0.023612,0.308468,0.418457,0.339446,0.396141,0.563838,0.569119,0.582336,0.544953,0.338853,-0.257583
run2,0.106431,0.113092,0.427299,0.193965,0.132235,0.343781,0.216198,0.24839,0.214533,0.373572,0.446589,0.344147,0.28847,0.273115,0.159816,-0.307956
run3,0.188298,0.078589,0.300996,0.120338,0.354757,-0.028691,0.262251,0.349522,0.219947,0.231213,0.408876,0.460399,0.436504,0.399696,0.338218,-0.244243
run4,0.268949,0.136038,0.275922,0.194083,0.3202,0.360556,0.021643,0.34403,0.030088,0.327267,0.517656,0.302604,0.399768,0.323469,0.158689,0.07466
run5,0.104183,0.22101,0.200702,0.19463,0.156902,0.050211,0.171541,0.181876,0.06259,0.286032,0.230426,0.270094,0.162094,0.131218,0.140505,-0.199057
run6,0.374461,0.222801,0.3855,0.113103,0.235778,0.227317,0.380617,0.416908,0.302856,0.3552,0.535434,0.59119,0.550654,0.506392,0.489739,0.07013
run7,0.367096,0.323647,0.464304,0.299793,0.492066,0.410256,0.506322,0.516421,0.393555,0.485554,0.595952,0.593436,0.597757,0.552648,0.35287,0.434099


In [7]:
gen_results2.T.to_csv('result/result_yield_CN_rmse.csv', header=False)
gen_results2.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
source,aats8p,aatsc3d,abcgg,atsc3d,atsc4i,bcutp1l,bertzct,estatevsa3,frnh0,ggi10,kappa2,kappa3,kier2,kier3,peoevsa6,vsaestate3
target,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN,Yield_CN
run0,29.532797,32.999928,32.49295,32.342216,26.560991,33.275074,32.131084,28.509153,32.119011,27.371428,31.628674,26.603148,30.72765,24.911091,33.717327,31.458908
run1,34.191616,33.283337,28.295744,32.142448,30.105951,36.896492,31.051325,28.475075,30.347858,29.016272,24.660288,24.51054,24.131685,25.188503,30.361473,41.873787
run2,39.362122,39.215134,31.512163,37.38448,38.789616,33.73175,36.865284,36.100285,36.904408,32.957157,30.976929,33.722343,35.124557,35.501534,38.168179,47.62236
run3,34.344646,36.59211,31.871368,35.753513,30.62122,38.663685,32.742752,30.745192,33.668427,33.424412,29.308941,28.002529,28.615812,29.535635,31.011177,42.521969
run4,33.550117,36.472683,33.389729,35.226166,32.35273,31.377718,38.81226,31.780607,38.644379,32.184113,27.252022,32.768764,30.400438,32.274845,35.991394,37.746006
run5,39.650799,36.974998,37.453876,37.595867,38.466373,40.827789,38.130974,37.892387,40.560848,35.398232,36.750854,35.791157,38.347755,39.047905,38.838634,45.873539
run6,31.038692,34.597301,30.763605,36.95837,34.307266,34.496639,30.885595,29.967108,32.767067,31.512934,26.748543,25.09211,26.306723,27.571926,28.033192,37.843147
run7,32.633419,33.734966,30.022915,34.324699,29.234592,31.50106,28.821428,28.525108,31.943987,29.4214,26.074144,26.155209,26.015827,27.435831,32.998131,30.857718
